In [15]:
from IPython.core.display import HTML
HTML("<script>Jupyter.notebook.kernel.restart()</script>")

In [11]:
import pandas as pd
import numpy as np
import warnings

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.tree import DecisionTreeClassifier
from sklearn.decomposition import PCA
from sklearn.utils import resample
from sklearn.model_selection import StratifiedKFold, cross_validate

warnings.filterwarnings('ignore')


In [12]:
class RotationForestClassifier(ClassifierMixin, BaseEstimator):
    
    # Etiqueta absoluta para que Scikit-Learn JAMÁS lo trate como regresor
    _estimator_type = "classifier"
    
    def __init__(self, n_estimators=50, max_features_per_subset=15, random_state=None):
        self.n_estimators = n_estimators
        self.max_features_per_subset = max_features_per_subset
        self.random_state = random_state
        
    def fit(self, X, y):
        np.random.seed(self.random_state)
        self.models = []
        self.rotation_matrices = []
        # Definición obligatoria para clasificadores en scikit-learn
        self.classes_ = np.unique(y) 
        
        n_features = X.shape[1]
        
        for i in range(self.n_estimators):
            features = np.random.permutation(n_features)
            subsets = [features[j:j + self.max_features_per_subset] 
                       for j in range(0, n_features, self.max_features_per_subset)]
            
            rotation_matrix = np.zeros((n_features, n_features))
            
            for subset in subsets:
                X_subset = X[:, subset]
                # Remuestreo (Bootstrap)
                X_sample = resample(X_subset, n_samples=int(X.shape[0]*0.75), random_state=self.random_state) 
                
                pca = PCA()
                pca.fit(X_sample)
                
                for j, feature_idx in enumerate(subset):
                    rotation_matrix[feature_idx, subset] = pca.components_[:, j]
                    
            X_rotated = np.dot(X, rotation_matrix)
            
            tree = DecisionTreeClassifier(random_state=self.random_state)
            tree.fit(X_rotated, y)
            
            self.models.append(tree)
            self.rotation_matrices.append(rotation_matrix)
            
        return self
    
    def predict_proba(self, X):
        probas = np.zeros((X.shape[0], len(self.classes_)))
        for tree, rot_mat in zip(self.models, self.rotation_matrices):
            X_rotated = np.dot(X, rot_mat)
            probas += tree.predict_proba(X_rotated)
        return probas / self.n_estimators
        
    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]

    # Método de seguridad para validadores de scikit-learn 1.4+
    def _more_tags(self):
        return {"estimator_type": "classifier"}


In [13]:

# 2. Carga y preprocesamiento

df = pd.read_csv('/kaggle/input/datasets/nico14h/metagenomica/dataset.csv', sep='\t')

df_pivot = df.pivot(index='Paciente', columns='Bacteria_Limpia', values='AbundanciaTaxonomica')
df_pivot = df_pivot.fillna(0) 

labels = df[['Paciente', 'Diagnosticos']].drop_duplicates().set_index('Paciente')
data = df_pivot.join(labels)

X = data.drop(columns=['Diagnosticos']).values
y = data['Diagnosticos'].values

# 3. Entrenamiento y Evaluación
rf = RotationForestClassifier(n_estimators=50, max_features_per_subset=15, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Ejecutamos el cross validation con n_jobs=1 para evitar errores de paralelización ocultos
results = cross_validate(rf, X, y, cv=cv, scoring=['accuracy', 'f1', 'roc_auc'], n_jobs=1)

# 4. Resultados finales
print("--- MÉTRICAS DEL ROTATION FOREST ---")
print(f"Accuracy : {np.mean(results['test_accuracy']):.4f} ± {np.std(results['test_accuracy']):.4f}")
print(f"F1 Score : {np.mean(results['test_f1']):.4f} ± {np.std(results['test_f1']):.4f}")
print(f"ROC AUC  : {np.mean(results['test_roc_auc']):.4f} ± {np.std(results['test_roc_auc']):.4f}")

--- MÉTRICAS DEL ROTATION FOREST ---
Accuracy : 0.7943 ± 0.0749
F1 Score : 0.8364 ± 0.0554
ROC AUC  : 0.9123 ± 0.0469


In [21]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
import warnings

warnings.filterwarnings('ignore')

# 1. Cargar y pivotear los datos (Melt -> Matriz ancha)
df = pd.read_csv('/kaggle/input/datasets/nico14h/metagenomica/dataset.csv', sep='\t')
df_pivot = df.pivot(index='Paciente', columns='Bacteria_Limpia', values='AbundanciaTaxonomica').fillna(0)

# Unir con la columna Diagnosticos
labels = df[['Paciente', 'Diagnosticos']].drop_duplicates().set_index('Paciente')
data = df_pivot.join(labels)

X_df = data.drop(columns=['Diagnosticos'])
X = X_df.values
y = data['Diagnosticos'].values
nombres_bacterias = X_df.columns

# 2. Entrenar y validar el Random Forest
# class_weight='balanced' es importante para datos de salud
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', max_depth=5, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

resultados = cross_validate(rf, X, y, cv=cv, scoring=['accuracy', 'f1', 'roc_auc'], n_jobs=1)

print("--- MÉTRICAS DEL RANDOM FOREST ---")
print(f"Accuracy : {np.mean(resultados['test_accuracy']):.4f} ± {np.std(resultados['test_accuracy']):.4f}")
print(f"F1 Score : {np.mean(resultados['test_f1']):.4f} ± {np.std(resultados['test_f1']):.4f}")
print(f"ROC AUC  : {np.mean(resultados['test_roc_auc']):.4f} ± {np.std(resultados['test_roc_auc']):.4f}")

# 3. Extraer Interpretabilidad Biológica (Feature Importance)
# Ajustamos sobre todo el dataset para extraer la firma bacteriana final
rf.fit(X, y)
importancias = rf.feature_importances_

df_importancias = pd.DataFrame({
    'Bacteria': nombres_bacterias,
    'Importancia_Predictiva': importancias
}).sort_values(by='Importancia_Predictiva', ascending=False)

# 4. Calcular el perfil clínico (Aumentado vs Disminuido en TEA)
medias = data.groupby('Diagnosticos').mean().T
medias.columns = ['Media_HC', 'Media_TEA']
medias['Diferencia'] = medias['Media_TEA'] - medias['Media_HC']
medias['Estado_en_TEA'] = np.where(medias['Diferencia'] > 0, 'Aumentada', 'Disminuida')

# Unir la importancia matemática con la dirección biológica
df_final = df_importancias.merge(medias[['Estado_en_TEA']], left_on='Bacteria', right_index=True)

print("\n--- TOP 20 BACTERIAS CLAVE PARA EL DIAGNÓSTICO ---")
# Filtramos bacterias con importancia 0 (las que el RF ignoró por completo)
df_final = df_final[df_final['Importancia_Predictiva'] > 0]
print(df_final.head(20).to_string(index=False))

--- MÉTRICAS DEL RANDOM FOREST ---
Accuracy : 0.8440 ± 0.0674
F1 Score : 0.8719 ± 0.0540
ROC AUC  : 0.9210 ± 0.0504

--- TOP 20 BACTERIAS CLAVE PARA EL DIAGNÓSTICO ---
                       Bacteria  Importancia_Predictiva Estado_en_TEA
     [Eubacterium]_hallii_group                0.065509     Aumentada
                     uncultured                0.048676     Aumentada
                   Faecalitalea                0.037746    Disminuida
                 Parasutterella                0.031347    Disminuida
   [Ruminococcus]_torques_group                0.030192     Aumentada
                    Collinsella                0.027714     Aumentada
                          Dorea                0.026525     Aumentada
                     Tyzzerella                0.024107    Disminuida
   Lachnospiraceae_ND3007_group                0.023897    Disminuida
                      Alistipes                0.023634     Aumentada
    Erysipelotrichaceae_UCG-003                0.020998     Au